# Satellite validation and calibration of the offshore hindcast

Compares the offshore hindcast at the goals (`offshore_waves`, e.g. WHACS) with
satellite altimetry, and calibrates it per goal:

0. **Partition energy** — the share of the bulk energy the hindcast partitions
   hold; the calibration closes them on the bulk (optional, `close_partitions`,
   default on; the steps below and the calibrated hindcast use the closed partitions).
1. **Passes at each goal** — altimeter observations within `radius_km` of each
   goal's hindcast node, in water deeper than `min_depth`, with good quality
   flags; one median Hs per satellite pass; the hindcast at the pass time.
2. **Calibration per goal** — directional calibration (wind sea and swell, per
   direction sector; Mínguez et al., 2011) fitted on `train_fraction` of the
   passes (75 % by default) and validated on the rest.
3. **Storm tail per goal** — one quantile map of the bulk Hs per goal, on its
   own passes; the bulk and every partition are scaled by the same ratio.
4. **Calibrated hindcast** — written as a new zarr, to add to `offshore_waves`
   (e.g. `WHACS_cal`) and use in 08 onwards like any other offshore source.
5. **Buoy check** — raw and calibrated Hs against deep buoys near the goals,
   which the calibration never saw.

Everything comes from the `satellite` block of the case JSON. Nothing here is
specific to one hindcast: any `offshore_waves` entry on `(time, goal)` with
`hs`, `dir` (and partitions `phs{i}`, `pdir{i}`) works.

**Prerequisites:** `07a_Offshore_WHACS.ipynb` or `07a_Offshore_ERA5.ipynb` (the goal
zarr) and the altimeter store (`satellite.data_path`). Method:
`bluemath_tk.waves.calibration` and `bluemath_tk.waves.altimetry`.

In [ ]:
import json
import os
import sys

import pandas as pd

# GDAL/PROJ data paths when the kernel was not started through `conda activate`
os.environ.setdefault("GDAL_DATA", os.path.join(sys.prefix, "share", "gdal"))
os.environ.setdefault("PROJ_DATA", os.path.join(sys.prefix, "share", "proj"))

from utils import calibration as cal
from utils import viz
from utils.config import open_offshore_waves

## Configuration

In [ ]:
offshore_name = "WHACS"  # offshore_waves entry to validate and calibrate (e.g. "ERA5")

cfg = cal.config()
print(json.dumps(cfg, indent=1))

## 0. Partition energy

The spectral reconstruction sums the goal contributions of each partition, so the
partitions must hold the bulk energy. A hindcast may write fewer systems than its
spectrum has: WHACS (WAVEWATCH III default partitioning) writes the wind sea and
the three largest swells, so the energy of any further swell is in `hs` but in no
`phs{i}` (and WAVEWATCH III also drops partitions below a minimum Hs). The table
gives, per goal, the share of the bulk energy the partitions hold, over every hour
and split by whether the last partition slot holds energy: when the loss comes from
systems that are not written, it sits in the hours with the last slot used.

Unless `close_partitions` is `false` in the `satellite` block (default `true`), the
calibration closes them: every step below reads the hindcast with each hour's
partitions scaled by `sqrt(hs² / Σ phs²)` (`scale_partitions_to_bulk`; periods,
directions and spreads kept), which hands the missing energy to the systems that
are there, in proportion to theirs. The calibrated hindcast (`*_cal`) is therefore
closed; the source entry keeps the partitions as the hindcast gives them (a
spectral reconstruction from it starts low by this share). A hindcast without
partitions has nothing to close (the table is empty and the calibration works on
the bulk Hs and direction).

In [ ]:
energy_share = cal.partition_energy_share(offshore_name)
energy_share.round(4)

## 1. Satellite passes at each goal

Cached in `outputs/calibration/{offshore_name}/pairs.csv`; `refresh=True`
recomputes them (e.g. after changing `radius_km` or `min_depth`).

In [ ]:
pairs = cal.satellite_pairs(offshore_name, refresh=False)
pairs.groupby("goal").size().rename("passes").to_frame().T

In [ ]:
fig = viz.plot_satellite_pairs(pairs, cal.goal_nodes(offshore_name), cfg["radius_km"])

## 2. Calibration per goal

Fitted on `train_fraction` of each goal's passes; the metrics below are on the
**test** passes, which the fit never saw (bias = hindcast − satellite).

In [ ]:
calibrations = cal.fit_calibration(offshore_name)
metrics = cal.calibration_metrics(offshore_name)
metrics[metrics["set"] == "test"].pivot(
    index="goal", columns="stage", values=["n", "bias", "rmse", "slope"]
).round(3)

In [ ]:
fig = viz.plot_calibration_metrics(metrics, subset="test")

Per goal: test passes before and after the calibration, and the Hs factor of
each direction sector for wind sea and swell (red above 1, blue below, grey =
not enough data, left at 1).

In [ ]:
for gid, calval in calibrations.items():
    calval.plot(title=f"{offshore_name} · goal {gid}")

## 3. Storm tail: quantile mapping per goal

A factor per sector fixes the level, not a tail that is too short: storms need
more than calm seas. One quantile map per goal, on that goal's passes, brings its
directionally calibrated Hs to the altimeter distribution, with a constant offset
beyond the goal's tail quantile. That quantile is the highest that keeps
`min_tail_samples` passes above it (default 5), at most `tail_quantile` (default
0.999): with one to a few thousand passes per goal it falls between about p99.4
and p99.9, so
each storm offset rests on a handful of passes. Neighbouring goals share passes
when their `radius_km` circles overlap.

The table gives, per goal, the passes, the tail quantile and the Hs of the
satellite and of each stage (`raw`, `hindcast` = after the directional step,
`mapped`) at a few probabilities; `n_above` is how many passes the value rests on.

**Fitted on all the passes** (train and test of section 2), so everything in
this section is in-sample: the deep buoys (section 5) and `10_Validate_Buoys`
are the independent check.

In [ ]:
mappings = cal.fit_quantile_mapping(offshore_name)
qm_pairs = cal.mapped_pairs(offshore_name)
pd.concat(
    {
        gid: m.quantiles(raw=qm_pairs.loc[qm_pairs["goal"] == gid, "hs_raw"].to_numpy())
        .assign(n=len(m.evaluated_samples()), tail_quantile=m.tail_quantile)
        for gid, m in mappings.items()
    },
    names=["goal"],
).round(4)

How to read the four panels (one figure per goal):

- **(a) the map**: the correction added to the directional Hs, as a function of
  it. Below the first fitted quantile (p1) it is a factor, between the fitted
  quantiles (dots) a linear interpolation, beyond the goal's tail quantile a constant
  offset: every storm above it gets the same extra metres, set by the few
  passes counted in the red band. The grey histogram shows how many passes
  support each part of the curve.
- **(b) Q-Q**: hindcast quantiles against satellite quantiles; on the 1:1 line
  the two distributions agree. The directional step moves the level, the map
  bends the tail.
- **(c) tail**: the same on an exceedance-probability (log) axis, where the
  storms are visible.
- **(d) pairs**: mapped Hs of each pass against its satellite Hs. The map
  matches distributions, not passes: compare the RMSE with the directional
  step in section 2.

In [ ]:
for gid, m in mappings.items():
    m.plot(
        raw=qm_pairs.loc[qm_pairs["goal"] == gid, "hs_raw"].to_numpy(),
        labels={"reference": "satellite", "hindcast": "directional"},
        title=f"{offshore_name} · goal {gid} · quantile mapping (tail p{100 * m.tail_quantile:.2f})",
    )

**All goals side by side**: Hs quantile error (hindcast − satellite) per goal for
the three stages. `mapped` sits near zero at every goal by construction (each map
is fitted on its own passes); what to look at is how much each goal needed
(`raw` and `directional`) and how the corrections change from one goal to the
next. At p99 each goal has only ~10–40 passes above, so expect noise there.

In [ ]:
fig = viz.plot_quantile_mapping_goals(qm_pairs)

## 4. Write the calibrated hindcast

Applies each goal's factors and its tail map to the whole record (partition Hs
times its sector factor, bulk Hs rescaled, then both by the tail ratio; periods
and directions unchanged) and writes
`*_cal.zarr` next to the source. Add the printed entry to `offshore_waves` in
the case JSON.

In [ ]:
entry = cal.write_calibrated_hindcast(offshore_name)
print(json.dumps(entry, indent=4))

**What the calibration does to the whole record**: exceedance curves of the full
hourly series at each goal, raw and calibrated, with the satellite passes. In
the grey band (rarer than one pass) no satellite checks the record: there the
calibrated tail is the shape of the hindcast tail plus the constant offset.

In [ ]:
raw_hs = open_offshore_waves(offshore_name)["hs"].load()
calibrated_hs = open_offshore_waves(entry=entry)["hs"].load()
fig = viz.plot_calibrated_record(raw_hs, calibrated_hs, pairs)

## 5. Independent check: deep buoys near the goals

The satellite fitted both steps; the buoys did not. Buoys deeper than
`buoy_min_depth` and within `buoy_max_distance_km` of their closest goal node
measure nearly the same sea as the hindcast node, so raw and calibrated Hs there
(closest goal, no propagation) show whether the calibration helps. Metrics on
the hours both have (bias = hindcast − buoy); `q{p}` is the error at that Hs
quantile, the tail check (do not bin errors by observed Hs: regression to the
mean overstates the storm bias).

In [ ]:
buoy_max_distance_km = 50  # to the closest goal node
buoy_min_depth = 24  # m, mesh depth at the buoy

buoy_sites = cal.buoy_check_sites(offshore_name, max_distance_km=buoy_max_distance_km, min_depth=buoy_min_depth)
buoy_sites.round(1)

In [ ]:
buoy_hs, buoy_estimates = cal.buoy_check(offshore_name, entry, buoy_sites)
buoy_metrics = cal.buoy_check_metrics(buoy_hs, buoy_estimates)
buoy_metrics.groupby("estimate")[["bias", "rmse", "r2", "q50", "q99", "q99.9"]].median().round(3)

In [ ]:
figs = viz.plot_buoy_check(buoy_hs, buoy_estimates, buoy_sites, buoy_metrics)

## Next

- Add the entry above to `offshore_waves` in the case JSON.
- `08_Dynamic_Cases_Generation.ipynb` with `offshore_name = "WHACS_cal"` (the
  dynamic runs must be forced with the calibrated hindcast too, so that 09
  compares like with like), then 09–12 with the same name.
- `10_Validate_Buoys.ipynb` with both names: the buoys are an independent check
  of the calibration.